# RLBench Dataset Analysis: Object Masks, Gripper Position, and Scene Graph

This notebook demonstrates how to:
- Extract object masks for specific objects (cups and Franka Panda)
- Calculate gripper position using mask-based methods
- Track gripper open/closed states
- Create videos from episode frames
- Build an interactive scene graph with object relationships over time
- Save the scene graph as JSON

**Note:** Task and object configuration is now loaded from `info.json` in the dataset directory.

## Setup

In [12]:
import sys
import numpy as np
import pickle
from PIL import Image as PILImage
from PIL import ImageDraw as PILImageDraw
import matplotlib.pyplot as plt
import cv2
import os
import json
import base64
from ipywidgets import *
import IPython.display as ipd
from IPython.display import HTML, clear_output
import warnings

# Add src to path for imports
sys.path.insert(0, '/workspace/src')
import utils.scene_graph_utils as utils
warnings.filterwarnings('ignore')

# ============================================================================
# DATASET CONFIGURATION
# ============================================================================
DATASET_PATH = '/workspace/datasets/rlbench'
TASK_NAME = 'stack_cups'
VARIATION = 2
EPISODE = 0
CAMERA = 'front'

# ============================================================================
# LOAD TASK CONFIGURATION FROM info.json
# ============================================================================
task_path = os.path.join(DATASET_PATH, TASK_NAME)
task_info = utils.load_task_info(task_path)

OBJECTS = task_info['object_names']
OBJECT_MAPPING = task_info['object_mapping']
RELATIONSHIP_TYPES = task_info['relationship_types']

print(f"Loaded task configuration from {task_path}/info.json")
print(f"Objects: {OBJECTS}")
print(f"Relationships: {RELATIONSHIP_TYPES}")
print(f"Object mapping: {OBJECT_MAPPING}")

# ============================================================================
# Derived paths
# ============================================================================
output_dir = f"{DATASET_PATH}/{TASK_NAME}/variation{VARIATION}"

Loaded task configuration from /workspace/datasets/rlbench/stack_cups/info.json
Objects: ['robot', 'cup_1', 'cup_2', 'cup_3']
Relationships: ['holding', 'stacked']
Object mapping: {31: 'robot', 34: 'robot', 35: 'robot', 101: 'cup_1', 106: 'cup_2', 116: 'cup_3'}


## Load Episode Data

In [13]:
# Load data
demo, image_data = utils.load_episode_data(TASK_NAME, VARIATION, EPISODE, DATASET_PATH=DATASET_PATH, CAMERA=CAMERA)

# Extract gripper states
gripper_states = utils.extract_gripper_states(demo)

print(f"Loaded episode with {len(demo)} observations")
print(f"RGB images: {len(image_data['rgb'])}")
print(f"Mask images: {len(image_data['mask'])}")
print(f"Demo length: {len(demo)} steps")
print(f"Gripper states extracted: {len(gripper_states)} frames")

# Find frames where gripper state changes
gripper_change_frames = []
for i in range(1, len(gripper_states)):
    if gripper_states[i] != gripper_states[i-1]:
        gripper_change_frames.append(i)
        
print(f"Gripper state changes at frames: {gripper_change_frames}")

# Debug: Check first mask path
if image_data['mask']:
    first_mask_path = image_data['mask'][0]
    print(f"First mask path: {first_mask_path}")
    print(f"Path exists: {os.path.exists(first_mask_path)}")
else:
    print("No mask images found!")

Loaded episode with 192 observations
RGB images: 192
Mask images: 192
Demo length: 192 steps
Gripper states extracted: 192 frames
Gripper state changes at frames: [52, 118, 143, 190]
First mask path: /workspace/datasets/rlbench/stack_cups/variation2/episodes/episode0/front_mask/0.png
Path exists: True


## Extract Object Masks

Extract masks for specific objects (cups and Franka Panda robot).

In [14]:
# ---------- Build per-object handle groups from OBJECT_MAPPING ----------
# handles_by_object: {'robot': [31, 34, 35], 'cup_1': [101], ...}
handles_by_object = {name: [] for name in OBJECTS}
for handle_id, obj_name in OBJECT_MAPPING.items():
    if obj_name in handles_by_object:
        handles_by_object[obj_name].append(handle_id)
    else:
        print(f"⚠ Handle {handle_id} maps to '{obj_name}' which is not in OBJECTS — ignored")
for name in OBJECTS:
    handles_by_object[name].sort()

classified_handles = set(OBJECT_MAPPING.keys())

# ---------- Scan all frames for every unique handle ----------
if image_data['mask']:
    print("Scanning all frames to find all objects...")
    all_objects = set()
    for mask_path in image_data['mask']:
        mask = utils.decode_mask_image(mask_path)
        all_objects.update([h for h in np.unique(mask) if h != 0])

    print(f"Found {len(all_objects)} unique handles: {sorted(all_objects)}")
    mapped = sorted(all_objects & classified_handles)
    unmapped = sorted(all_objects - classified_handles)
    print(f"\n✓ Classified ({len(mapped)}): {mapped}")
    if unmapped:
        print(f"⚠ Unclassified ({len(unmapped)}): {unmapped}  — add to info.json if needed")
    print(f"\nObject groups:")
    for name in OBJECTS:
        print(f"  {name}: handles = {handles_by_object[name]}")
else:
    all_objects = set()
    print("No mask images found!")

Scanning all frames to find all objects...
Found 18 unique handles: [10, 31, 34, 35, 39, 40, 41, 42, 43, 44, 45, 46, 48, 52, 55, 101, 106, 116]

✓ Classified (6): [31, 34, 35, 101, 106, 116]
⚠ Unclassified (12): [10, 39, 40, 41, 42, 43, 44, 45, 46, 48, 52, 55]  — add to info.json if needed

Object groups:
  robot: handles = [31, 34, 35]
  cup_1: handles = [101]
  cup_2: handles = [106]
  cup_3: handles = [116]


## Extract Gripper Position from Masks

Calculate gripper position using the center of mass of robot mask pixels.

In [15]:
# Extract positions
positions = utils.extract_positions_over_time(image_data, OBJECTS, handles_by_object, start_frame=20)

print(f"Extracted positions for {len(positions['frames'])} frames (from frame 20)")
print(f"Frame range: {positions['frames'][0]} to {positions['frames'][-1]}")

Extracted positions for 172 frames (from frame 20)
Frame range: 20 to 191


## Create Episode Video

In [16]:
# Interactive relationship editor — fully driven by OBJECTS

start_frame = positions['frames'][0] if positions['frames'] else 0

scene_graph = {
    'objects': {name: {'type': name} for name in OBJECTS},
    'frames': []
}

for pos_idx, frame_id in enumerate(positions['frames']):
    frame_data = {
        'frame_id': frame_id,
        'timestamp': frame_id / 10.0,
        'objects': {},
        'relationships': []
    }
    for name in OBJECTS:
        pos = positions[name][pos_idx]
        frame_data['objects'][name] = {
            'position': list(pos) if pos is not None else [0, 0],
            'visible': pos is not None
        }
    if frame_id < len(gripper_states):
        frame_data['gripper_closed'] = gripper_states[frame_id]
    
    scene_graph['frames'].append(frame_data)

print(f"Built scene graph: {len(scene_graph['frames'])} frames, objects = {OBJECTS}")

# Overlay colour palette for visualisation (one colour per object)
_overlay_palette = [
    [255, 50, 50], [50, 120, 255], [50, 220, 50], [255, 180, 30],
    [180, 50, 255], [0, 220, 220], [255, 105, 180], [160, 110, 50],
]
_obj_colors = {}
for i, name in enumerate(OBJECTS):
    _obj_colors[name] = _overlay_palette[i % len(_overlay_palette)]

if not scene_graph['frames']:
    print("⚠️ No frames. Run position extraction first.")
else:
    frame_slider = IntSlider(value=0, min=0, max=len(scene_graph['frames'])-1,
                             description='Frame:', continuous_update=False)
    frame_input = BoundedIntText(value=0, min=0, max=len(scene_graph['frames'])-1,
                                 description='Go to:', step=1)
    relationship_type = Dropdown(options=RELATIONSHIP_TYPES, value=RELATIONSHIP_TYPES[0],
                                 description='Relationship:')
    object1_dropdown = Dropdown(options=OBJECTS, description='Object 1:')
    object2_dropdown = Dropdown(options=OBJECTS, description='Object 2:')

    add_relationship_btn = Button(description='Add Relationship')
    clear_relationships_btn = Button(description='Clear Frame Rels')
    save_graph_btn = Button(description='Save Scene Graph')
    existing_relationships_dropdown = Dropdown(options=[], description='Existing Rel:')
    clear_specific_relationship_btn = Button(description='Clear Selected Rel', button_style='warning')

    # Gripper change navigation
    prev_gripper_btn = Button(description='← Prev Gripper Change', button_style='info')
    next_gripper_btn = Button(description='Next Gripper Change →', button_style='info')

    output_area = Output()

    def _update_rel_dropdown(idx):
        fd = scene_graph['frames'][idx]
        opts = [(f"{r['object1']} --{r['type']}--> {r['object2']}", r) for r in fd['relationships']]
        existing_relationships_dropdown.options = opts
        existing_relationships_dropdown.value = opts[0][1] if opts else None

    def update_frame_display(idx):
        with output_area:
            clear_output(wait=True)
            fd = scene_graph['frames'][idx]
            fid = fd['frame_id']
            
            is_gripper_change = fid in gripper_change_frames
            gripper_state = "CLOSED" if fd.get('gripper_closed', False) else "OPEN"
            gripper_indicator = "🔴 GRIPPER STATE CHANGE! " if is_gripper_change else ""

            print(f"=== Frame {fid} (idx {idx}, t={fd['timestamp']:.2f}s) ===")
            print(f"{gripper_indicator}Gripper: {gripper_state}")
            print("\nObjects:")
            for name in OBJECTS:
                od = fd['objects'].get(name, {'position': [0,0], 'visible': False})
                vis = '✓' if od['visible'] else '✗'
                print(f"  {vis} {name}: ({od['position'][0]}, {od['position'][1]})")

            print("\nRelationships:")
            if fd['relationships']:
                for r in fd['relationships']:
                    print(f"  {r['object1']} --{r['type']}--> {r['object2']}")
            else:
                print("  (none)")
            _update_rel_dropdown(idx)

            if fid < len(image_data['rgb']):
                rgb = cv2.imread(image_data['rgb'][fid])
                rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
                mask = utils.decode_mask_image(image_data['mask'][fid])

                overlay = np.zeros_like(rgb, dtype=np.uint8)
                for name in OBJECTS:
                    color = _obj_colors[name]
                    for h in handles_by_object[name]:
                        overlay[mask == h] = color

                blended = cv2.addWeighted(rgb, 0.7, overlay, 0.3, 0)

                for name in OBJECTS:
                    od = fd['objects'].get(name)
                    if od and od['visible']:
                        x, y = int(od['position'][0]), int(od['position'][1])
                        cv2.circle(blended, (x, y), 8, (255, 255, 255), 2)
                        cv2.putText(blended, name, (x+10, y-10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                for r in fd['relationships']:
                    o1, o2 = r['object1'], r['object2']
                    if o1 in fd['objects'] and o2 in fd['objects']:
                        p1 = fd['objects'][o1]['position']
                        p2 = fd['objects'][o2]['position']
                        pt1, pt2 = (int(p1[0]), int(p1[1])), (int(p2[0]), int(p2[1]))
                        cv2.arrowedLine(blended, pt1, pt2, (0, 255, 255), 2, tipLength=0.1)
                        mx, my = (pt1[0]+pt2[0])//2, (pt1[1]+pt2[1])//2
                        cv2.putText(blended, r['type'], (mx+5, my-5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

                # Gripper state change indicator
                if is_gripper_change:
                    h, w = blended.shape[:2]
                    cv2.rectangle(blended, (0, 0), (w-1, h-1), (255, 0, 0), 8)
                    cv2.putText(blended, f"GRIPPER {gripper_state}!", (20, 80),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

                plt.figure(figsize=(12, 8))
                plt.imshow(blended)
                title = f'Frame {fid} — Mask Overlay + Scene Graph'
                if is_gripper_change:
                    title += f' [GRIPPER {gripper_state}]'
                plt.title(title)
                plt.axis('off')
                plt.tight_layout()
                plt.show()

    def add_relationship(b):
        idx = frame_slider.value
        o1, o2, rt = object1_dropdown.value, object2_dropdown.value, relationship_type.value
        if o1 == o2:
            with output_area: print("❌ Cannot relate an object to itself")
            return
        for i in range(idx, len(scene_graph['frames'])):
            rels = scene_graph['frames'][i]['relationships']
            rels[:] = [r for r in rels if not (r['object1'] == o1 and r['object2'] == o2)]
            rels.append({'object1': o1, 'object2': o2, 'type': rt})
        update_frame_display(idx)

    def clear_relationships(b):
        idx = frame_slider.value
        scene_graph['frames'][idx]['relationships'] = []
        update_frame_display(idx)

    def clear_specific_relationship(b):
        idx = frame_slider.value
        sel = existing_relationships_dropdown.value
        if sel is None: return
        o1, o2 = sel['object1'], sel['object2']
        for i in range(idx, len(scene_graph['frames'])):
            scene_graph['frames'][i]['relationships'][:] = [
                r for r in scene_graph['frames'][i]['relationships']
                if not (r['object1'] == o1 and r['object2'] == o2)]
        update_frame_display(idx)

    def save_scene_graph(b):
        fpath = os.path.join(output_dir, f'{TASK_NAME}_scene_graph.json')
        with open(fpath, 'w') as f:
            json.dump(scene_graph, f, indent=2)
        with output_area: print(f"✓ Scene graph saved to {fpath}")

    def on_slider_change(change):
        frame_input.value = change['new']
        update_frame_display(change['new'])
    
    def on_input_change(change):
        frame_slider.value = change['new']

    def _goto_prev_gripper(b):
        current_frame = scene_graph['frames'][frame_slider.value]['frame_id']
        prev_frames = [f for f in gripper_change_frames if f < current_frame]
        if prev_frames:
            target = prev_frames[-1]
            # Find idx in scene_graph
            for i, fd in enumerate(scene_graph['frames']):
                if fd['frame_id'] == target:
                    frame_slider.value = i
                    break

    def _goto_next_gripper(b):
        current_frame = scene_graph['frames'][frame_slider.value]['frame_id']
        next_frames = [f for f in gripper_change_frames if f > current_frame]
        if next_frames:
            target = next_frames[0]
            for i, fd in enumerate(scene_graph['frames']):
                if fd['frame_id'] == target:
                    frame_slider.value = i
                    break

    frame_slider.observe(on_slider_change, names='value')
    frame_input.observe(on_input_change, names='value')
    add_relationship_btn.on_click(add_relationship)
    clear_relationships_btn.on_click(clear_relationships)
    clear_specific_relationship_btn.on_click(clear_specific_relationship)
    save_graph_btn.on_click(save_scene_graph)
    prev_gripper_btn.on_click(_goto_prev_gripper)
    next_gripper_btn.on_click(_goto_next_gripper)

    print(f"Scene graph: {len(scene_graph['frames'])} frames, objects = {OBJECTS}")
    print(f"Output dir: {output_dir}")
    print(f"Gripper state change frames: {gripper_change_frames}\n")

    controls = VBox([
        HBox([frame_slider, frame_input]),
        HBox([prev_gripper_btn, next_gripper_btn]),
        HBox([object1_dropdown, relationship_type, object2_dropdown, add_relationship_btn]),
        HBox([existing_relationships_dropdown, clear_specific_relationship_btn]),
        HBox([clear_relationships_btn, save_graph_btn])
    ])
    ipd.display(controls)
    ipd.display(output_area)
    update_frame_display(0)

Built scene graph: 172 frames, objects = ['robot', 'cup_1', 'cup_2', 'cup_3']
Scene graph: 172 frames, objects = ['robot', 'cup_1', 'cup_2', 'cup_3']
Output dir: /workspace/datasets/rlbench/stack_cups/variation2
Gripper state change frames: [52, 118, 143, 190]



Output()

In [17]:
# Load saved scene graph and create both videos + object color map
scene_graph_file = os.path.join(output_dir, f'{TASK_NAME}_scene_graph.json')

if os.path.exists(scene_graph_file):
    with open(scene_graph_file, 'r') as f:
        loaded_scene_graph = json.load(f)
    print(f"Loaded scene graph: {len(loaded_scene_graph['frames'])} frames")

    result = utils.create_videos(
        image_data, OBJECTS, OBJECT_MAPPING, handles_by_object,
        loaded_scene_graph, output_dir, start_frame=20
    )

    if result:
        overlay_path, mask_path, json_path = result

        # Use HTML widget for reliable video embedding (works with h264 mp4)
        import base64

        def _embed_video(path, title):
            """Embed video as base64 HTML for reliable notebook display."""
            with open(path, 'rb') as f:
                b64 = base64.b64encode(f.read()).decode('utf-8')
            html = f'''
            <p><strong>{title}</strong></p>
            <video controls width="640" height="400">
              <source src="data:video/mp4;base64,{b64}" type="video/mp4">
              Your browser does not support the video tag.
            </video>
            '''
            ipd.display(ipd.HTML(html))

        print("\n--- Overlay Video (RGB + mask + scene graph) ---")
        _embed_video(overlay_path, "Overlay Video")

        print("\n--- Mask Video (vivid colours for visualization) ---")
        _embed_video(mask_path, "Mask Video (Vivid)")

        print("\n--- Object Color Map ---")
        with open(json_path, 'r') as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("Failed to create videos")
else:
    print(f"Scene graph not found: {scene_graph_file}")
    print("Save the scene graph first using the 'Save Scene Graph' button.")

Loaded scene graph: 172 frames
✓ object_color_map.json saved
  obj_1: robot, handles=[31, 34, 35], mask=(0,0,1)
  obj_2: cup_1, handles=[101], mask=(0,0,2)
  obj_3: cup_2, handles=[106], mask=(0,0,3)
  obj_4: cup_3, handles=[116], mask=(0,0,4)


Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.
Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.
Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


✓ Encoded mask: /workspace/datasets/rlbench/stack_cups/variation2/episode_mask_encoded.mp4 (172 frames) — pixel=(0,0,object_index)

✓ Overlay video: /workspace/datasets/rlbench/stack_cups/variation2/episode_overlay.mp4 (172 frames)
✓ Mask video (vivid): /workspace/datasets/rlbench/stack_cups/variation2/episode_mask.mp4 (172 frames) — for visualization
  Mask encoding in encoded file: pixel = (0, 0, object_index). Background = (0,0,0).

--- Overlay Video (RGB + mask + scene graph) ---



--- Mask Video (vivid colours for visualization) ---



--- Object Color Map ---
{
  "obj_1": {
    "name": "robot",
    "index": 1,
    "handle_ids": [
      31,
      34,
      35
    ],
    "mask_color_rgb": [
      0,
      0,
      1
    ]
  },
  "obj_2": {
    "name": "cup_1",
    "index": 2,
    "handle_ids": [
      101
    ],
    "mask_color_rgb": [
      0,
      0,
      2
    ]
  },
  "obj_3": {
    "name": "cup_2",
    "index": 3,
    "handle_ids": [
      106
    ],
    "mask_color_rgb": [
      0,
      0,
      3
    ]
  },
  "obj_4": {
    "name": "cup_3",
    "index": 4,
    "handle_ids": [
      116
    ],
    "mask_color_rgb": [
      0,
      0,
      4
    ]
  }
}
